# Module 1.1: Hidden Markov Model Definition and Framework

## Learning Objectives
By completing this notebook, you will:
1. Understand the five-tuple HMM definition and mathematical notation
2. Distinguish between observable outputs and hidden states
3. Grasp the three fundamental problems in HMM theory
4. Implement basic HMM structure and parameter representation
5. Visualize HMM components and their relationships

## Prerequisites
- Basic probability theory (conditional probability, joint distributions)
- Markov chains (transition matrices, stationary distributions)
- Linear algebra (matrix operations)
- Python programming basics

## Estimated Time: 45 minutes

---

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The HMM Five-Tuple Definition

A **Hidden Markov Model** is formally defined by the five-tuple:

$$\lambda = (S, O, A, B, \pi)$$

Where:

1. **S = {s₁, s₂, ..., sₖ}**: Set of K hidden states
2. **O = {o₁, o₂, ..., oₘ}**: Set of M possible observations
3. **A**: State transition matrix (K × K)
   - $A_{ij} = P(q_{t+1} = s_j | q_t = s_i)$
4. **B**: Emission/observation matrix (K × M)
   - $B_{ij} = P(o_t = o_j | q_t = s_i)$
5. **π**: Initial state distribution (K × 1)
   - $\pi_i = P(q_1 = s_i)$

### Key Assumptions

1. **First-order Markov**: $P(q_t | q_{t-1}, q_{t-2}, ..., q_1) = P(q_t | q_{t-1})$
2. **Output independence**: $P(o_t | q_1, ..., q_T, o_1, ..., o_{t-1}, o_{t+1}, ..., o_T) = P(o_t | q_t)$
3. **Stationarity**: Transition and emission probabilities don't change over time

## 2. Example: Weather Model

A classic HMM example to build intuition:

**Scenario**: You're locked in a room. You can't see outside (hidden states), but someone tells you their activity each day (observations).

- **Hidden States**: {Sunny, Rainy}
- **Observations**: {Walk, Shop, Clean}

The weather (hidden state) influences the activity (observation).

In [ ]:
# Purpose: Define a simple weather HMM with explicit parameter structure
# Key Concept: HMM as a complete probability model

class DiscreteHMM:
    """
    Discrete Hidden Markov Model.
    
    Parameters
    ----------
    states : list of str
        Hidden state labels
    observations : list of str
        Observation symbols
    A : ndarray (K, K)
        Transition matrix
    B : ndarray (K, M)
        Emission matrix
    pi : ndarray (K,)
        Initial state distribution
    """
    
    def __init__(self, states, observations, A, B, pi):
        self.states = states
        self.observations = observations
        self.K = len(states)  # Number of states
        self.M = len(observations)  # Number of observations
        
        # Validate shapes
        assert A.shape == (self.K, self.K), f"A must be {self.K}x{self.K}"
        assert B.shape == (self.K, self.M), f"B must be {self.K}x{self.M}"
        assert len(pi) == self.K, f"pi must have length {self.K}"
        
        # Validate probability constraints
        assert np.allclose(A.sum(axis=1), 1), "A rows must sum to 1"
        assert np.allclose(B.sum(axis=1), 1), "B rows must sum to 1"
        assert np.isclose(pi.sum(), 1), "pi must sum to 1"
        
        self.A = A
        self.B = B
        self.pi = pi
        
        # Create lookup dictionaries
        self.state_to_idx = {s: i for i, s in enumerate(states)}
        self.obs_to_idx = {o: i for i, o in enumerate(observations)}
    
    def __repr__(self):
        return f"DiscreteHMM(states={self.K}, observations={self.M})"
    
    def display_parameters(self):
        """Pretty print HMM parameters."""
        print("="*70)
        print("HIDDEN MARKOV MODEL PARAMETERS")
        print("="*70)
        
        print(f"\n1. STATES (K={self.K}): {self.states}")
        print(f"\n2. OBSERVATIONS (M={self.M}): {self.observations}")
        
        print(f"\n3. INITIAL DISTRIBUTION (π):")
        for i, state in enumerate(self.states):
            print(f"   P(q₁ = {state:6s}) = {self.pi[i]:.3f}")
        
        print(f"\n4. TRANSITION MATRIX (A):")
        print(f"   {'':8s} " + " ".join(f"{s:8s}" for s in self.states))
        for i, state_i in enumerate(self.states):
            row = " ".join(f"{self.A[i,j]:8.3f}" for j in range(self.K))
            print(f"   {state_i:8s} {row}")
        
        print(f"\n5. EMISSION MATRIX (B):")
        print(f"   {'':8s} " + " ".join(f"{o:8s}" for o in self.observations))
        for i, state in enumerate(self.states):
            row = " ".join(f"{self.B[i,j]:8.3f}" for j in range(self.M))
            print(f"   {state:8s} {row}")
        
        print("\n" + "="*70)

# Define weather HMM
states = ['Sunny', 'Rainy']
observations = ['Walk', 'Shop', 'Clean']

# Transition matrix: weather tends to persist
A = np.array([
    [0.7, 0.3],  # Sunny → [Sunny, Rainy]
    [0.4, 0.6]   # Rainy → [Sunny, Rainy]
])

# Emission matrix: activities depend on weather
B = np.array([
    [0.6, 0.3, 0.1],  # Sunny → [Walk, Shop, Clean]
    [0.1, 0.4, 0.5]   # Rainy → [Walk, Shop, Clean]
])

# Initial distribution: slightly more likely to start sunny
pi = np.array([0.6, 0.4])

# Create HMM
weather_hmm = DiscreteHMM(states, observations, A, B, pi)
weather_hmm.display_parameters()

## 3. Visualizing the HMM Structure

HMMs have two interconnected layers:
1. **Hidden state chain**: Markov process over states
2. **Observation sequence**: Emissions from hidden states

In [ ]:
# Purpose: Visualize HMM architecture and temporal dynamics
# Key Concept: Hidden states generate observations

def visualize_hmm_structure(hmm, T=5):
    """
    Create visualization of HMM temporal structure.
    
    Parameters
    ----------
    hmm : DiscreteHMM
        The HMM to visualize
    T : int
        Number of time steps to show
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Hidden states layer (top)
    state_y = 3.5
    obs_y = 1.5
    
    for t in range(T):
        x = t * 2.5 + 1
        
        # Draw hidden state node
        circle = plt.Circle((x, state_y), 0.4, 
                           color='lightblue', ec='black', linewidth=2)
        ax.add_patch(circle)
        ax.text(x, state_y, f'q{t+1}', ha='center', va='center', 
               fontsize=12, fontweight='bold')
        
        # Draw observation node
        square = FancyBboxPatch((x-0.4, obs_y-0.4), 0.8, 0.8,
                               boxstyle="round,pad=0.05",
                               facecolor='lightcoral', 
                               edgecolor='black', linewidth=2)
        ax.add_patch(square)
        ax.text(x, obs_y, f'o{t+1}', ha='center', va='center',
               fontsize=12, fontweight='bold')
        
        # Draw emission arrow (state → observation)
        arrow = FancyArrowPatch((x, state_y-0.5), (x, obs_y+0.5),
                               arrowstyle='->', mutation_scale=20,
                               color='gray', linewidth=2)
        ax.add_patch(arrow)
        
        # Draw transition arrow to next state
        if t < T - 1:
            arrow = FancyArrowPatch((x+0.4, state_y), (x+2.1, state_y),
                                   arrowstyle='->', mutation_scale=20,
                                   color='blue', linewidth=2.5)
            ax.add_patch(arrow)
    
    # Labels
    ax.text(-0.5, state_y, 'Hidden\nStates', ha='center', va='center',
           fontsize=14, fontweight='bold', color='blue')
    ax.text(-0.5, obs_y, 'Observations', ha='center', va='center',
           fontsize=14, fontweight='bold', color='red')
    
    # Annotations
    ax.text((T-1)*2.5/2 + 1, 2.7, 'Emission probabilities (B)',
           ha='center', fontsize=11, style='italic', color='gray')
    ax.text((T-1)*2.5/2 + 1, state_y+0.8, 'Transition probabilities (A)',
           ha='center', fontsize=11, style='italic', color='blue')
    
    ax.set_xlim(-1.5, T*2.5 + 0.5)
    ax.set_ylim(0.5, 4.5)
    ax.axis('off')
    ax.set_title('Hidden Markov Model Structure', fontsize=16, pad=20)
    
    plt.tight_layout()
    plt.show()

visualize_hmm_structure(weather_hmm, T=5)

## 4. The Three Fundamental Problems

Every application of HMMs involves solving one of these three problems:

### Problem 1: Evaluation (Likelihood)
**Given**: Model λ = (A, B, π) and observation sequence O = o₁, o₂, ..., oₜ  
**Find**: P(O | λ)  
**Use Case**: Model comparison, anomaly detection
**Solution**: Forward-Backward algorithm

### Problem 2: Decoding (State Sequence)
**Given**: Model λ and observation sequence O  
**Find**: Most likely state sequence Q = q₁, q₂, ..., qₜ  
**Use Case**: Regime identification, speech recognition
**Solution**: Viterbi algorithm

### Problem 3: Learning (Parameter Estimation)
**Given**: Observation sequence O (and possibly number of states K)  
**Find**: Model parameters λ* that maximize P(O | λ)  
**Use Case**: Training from data
**Solution**: Baum-Welch algorithm (EM)

In [ ]:
# Purpose: Illustrate the three problems with concrete examples
# Key Concept: Different questions require different algorithms

print("THE THREE FUNDAMENTAL PROBLEMS OF HMMs")
print("="*70)

# Example observation sequence
obs_sequence = ['Walk', 'Walk', 'Shop', 'Clean', 'Clean']
print(f"\nObservation sequence: {obs_sequence}")
print(f"Length: {len(obs_sequence)} days\n")

print("PROBLEM 1: EVALUATION")
print("-" * 70)
print("Question: How likely is this observation sequence?")
print(f"Goal: Compute P(O | λ)")
print("Application: Is this sequence typical for our weather model?")
print("Algorithm: Forward algorithm (Module 2)\n")

print("PROBLEM 2: DECODING")
print("-" * 70)
print("Question: What was the weather on each day?")
print(f"Goal: Find most likely Q = q₁, q₂, ..., q₅")
print("Application: Infer hidden states from observations")
print("Algorithm: Viterbi algorithm (Module 2)\n")

print("PROBLEM 3: LEARNING")
print("-" * 70)
print("Question: What are the true A, B, π parameters?")
print(f"Goal: Find λ* = argmax P(O | λ)")
print("Application: Learn model from observed activity patterns")
print("Algorithm: Baum-Welch (EM) algorithm (Module 2)")

print("\n" + "="*70)

## 5. Financial Application Preview

HMMs are widely used in finance for regime detection:

- **Hidden States**: Market regimes (Bull, Bear, Sideways)
- **Observations**: Asset returns, volatility, trading volume
- **Applications**: Tactical allocation, risk management, trading signals

In [ ]:
# Purpose: Preview financial HMM structure
# Key Concept: HMMs model regime-dependent return distributions

# Define financial regime HMM (simplified)
fin_states = ['Bull', 'Bear', 'Sideways']
# For continuous returns, we'd use Gaussian emissions
# Here we discretize for illustration
fin_obs = ['High+', 'Low+', 'Neutral', 'Low-', 'High-']

# Transition matrix: regimes persist but eventually switch
A_fin = np.array([
    [0.85, 0.05, 0.10],  # Bull tends to stay Bull
    [0.10, 0.80, 0.10],  # Bear tends to stay Bear
    [0.20, 0.15, 0.65]   # Sideways is transitional
])

# Emission matrix: returns depend on regime
B_fin = np.array([
    [0.40, 0.30, 0.20, 0.05, 0.05],  # Bull: mostly positive
    [0.05, 0.10, 0.15, 0.35, 0.35],  # Bear: mostly negative
    [0.15, 0.25, 0.30, 0.20, 0.10]   # Sideways: centered
])

pi_fin = np.array([0.5, 0.2, 0.3])

market_hmm = DiscreteHMM(fin_states, fin_obs, A_fin, B_fin, pi_fin)

print("FINANCIAL MARKET REGIME HMM\n")
market_hmm.display_parameters()

print("\nINTERPRETATION:")
print("- Bull market: High probability of positive returns")
print("- Bear market: High probability of negative returns")
print("- Sideways: Mixed returns, transitional regime")
print("- All regimes show persistence (high diagonal in A)")

## Exercise 1: Verify Probability Constraints

**Task:** Implement a function to verify that HMM parameters satisfy probability constraints:
1. All elements in A, B, π are non-negative
2. Each row of A sums to 1
3. Each row of B sums to 1
4. π sums to 1

**Expected Output:** Boolean indicating validity and list of violations.

In [ ]:
# YOUR CODE HERE
# ---------------

def verify_hmm_constraints(A, B, pi, tolerance=1e-6):
    """
    Verify HMM probability constraints.
    
    Parameters
    ----------
    A : ndarray (K, K)
        Transition matrix
    B : ndarray (K, M)
        Emission matrix
    pi : ndarray (K,)
        Initial distribution
    tolerance : float
        Numerical tolerance for sum checks
    
    Returns
    -------
    valid : bool
        True if all constraints satisfied
    violations : list of str
        List of constraint violations
    """
    violations = []
    
    # YOUR IMPLEMENTATION HERE
    # Check:
    # 1. Non-negativity of all elements
    # 2. Row sums of A
    # 3. Row sums of B
    # 4. Sum of pi
    
    return None, None

# Test on weather HMM
valid, violations = verify_hmm_constraints(weather_hmm.A, weather_hmm.B, weather_hmm.pi)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert valid is not None, "Don't forget to return validity!"
    assert violations is not None, "Don't forget to return violations!"
    assert isinstance(valid, bool), "valid should be boolean"
    assert isinstance(violations, list), "violations should be list"
    
    # Weather HMM should be valid
    assert valid == True, "Weather HMM should be valid!"
    assert len(violations) == 0, "Weather HMM should have no violations"
    
    # Test invalid matrices
    A_bad = np.array([[0.8, 0.3], [0.4, 0.6]])  # First row sums > 1
    B_test = np.array([[0.5, 0.5], [0.5, 0.5]])
    pi_test = np.array([0.5, 0.5])
    
    valid_bad, viol_bad = verify_hmm_constraints(A_bad, B_test, pi_test)
    assert valid_bad == False, "Should detect invalid A matrix"
    assert len(viol_bad) > 0, "Should report violations"
    
    print("✅ Exercise 1 passed!")
    print("\nWeather HMM validation:")
    print(f"  Valid: {valid}")
    print(f"  Violations: {violations if violations else 'None'}")

test_exercise_1()

## Exercise 2: Compute Joint Probability

**Task:** Given a specific state sequence and observation sequence, compute the joint probability:

$$P(Q, O | \lambda) = \pi_{q_1} \cdot b_{q_1}(o_1) \cdot \prod_{t=2}^{T} a_{q_{t-1}, q_t} \cdot b_{q_t}(o_t)$$

**Example**: 
- States: [Sunny, Sunny, Rainy]
- Observations: [Walk, Shop, Clean]

**Expected Output:** Joint probability value.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_joint_probability(hmm, state_seq, obs_seq):
    """
    Compute P(Q, O | λ) for given sequences.
    
    Parameters
    ----------
    hmm : DiscreteHMM
        The HMM model
    state_seq : list of str
        State sequence
    obs_seq : list of str
        Observation sequence
    
    Returns
    -------
    prob : float
        Joint probability
    """
    # YOUR IMPLEMENTATION HERE
    # Steps:
    # 1. Convert state/obs names to indices
    # 2. Start with π[q₁] * B[q₁, o₁]
    # 3. Multiply by A[q_{t-1}, q_t] * B[q_t, o_t] for t=2..T
    
    return None

# Test cases
test_states = ['Sunny', 'Sunny', 'Rainy']
test_obs = ['Walk', 'Shop', 'Clean']

joint_prob = compute_joint_probability(weather_hmm, test_states, test_obs)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert joint_prob is not None, "Don't forget to compute joint probability!"
    assert isinstance(joint_prob, (float, np.floating)), "Should return float"
    assert 0 <= joint_prob <= 1, "Probability must be in [0, 1]"
    
    # Verify computation manually for small case
    # P = π[Sunny] * B[Sunny,Walk] * A[Sunny,Sunny] * B[Sunny,Shop] * 
    #     A[Sunny,Rainy] * B[Rainy,Clean]
    expected = (weather_hmm.pi[0] * weather_hmm.B[0, 0] *
                weather_hmm.A[0, 0] * weather_hmm.B[0, 1] *
                weather_hmm.A[0, 1] * weather_hmm.B[1, 2])
    
    assert np.isclose(joint_prob, expected, rtol=1e-5), \
        f"Expected {expected:.6f}, got {joint_prob:.6f}"
    
    print("✅ Exercise 2 passed!")
    print(f"\nJoint probability: {joint_prob:.8f}")
    print(f"\nInterpretation:")
    print(f"  This specific state-observation sequence has")
    print(f"  probability {joint_prob:.8f} ≈ {joint_prob*100:.4f}%")

test_exercise_2()

## Exercise 3: Create Custom HMM

**Task:** Design your own HMM for a trading strategy with:
- **States**: {Trending, Mean-Reverting, High-Vol}
- **Observations**: {Strong-Up, Weak-Up, Flat, Weak-Down, Strong-Down}

Define reasonable A, B, and π matrices, then validate them.

**Expected Output:** Valid HMM with trading regime interpretation.

In [ ]:
# YOUR CODE HERE
# ---------------

# Define your trading HMM
trading_states = ['Trending', 'Mean-Reverting', 'High-Vol']
trading_obs = ['Strong-Up', 'Weak-Up', 'Flat', 'Weak-Down', 'Strong-Down']

# YOUR MATRICES HERE
A_trading = None  # Replace with your 3x3 matrix
B_trading = None  # Replace with your 3x5 matrix
pi_trading = None # Replace with your 3-element vector

# Create HMM (if matrices defined)
if A_trading is not None:
    trading_hmm = DiscreteHMM(trading_states, trading_obs, 
                              A_trading, B_trading, pi_trading)
else:
    trading_hmm = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert trading_hmm is not None, "Don't forget to create trading HMM!"
    assert trading_hmm.K == 3, "Should have 3 states"
    assert trading_hmm.M == 5, "Should have 5 observations"
    
    # Validate constraints
    valid, violations = verify_hmm_constraints(
        trading_hmm.A, trading_hmm.B, trading_hmm.pi
    )
    assert valid, f"Trading HMM has violations: {violations}"
    
    # Check that states make sense
    # Trending should have directional bias
    # Mean-reverting should have higher Flat probability
    # High-vol should have higher extreme probabilities
    
    print("✅ Exercise 3 passed!")
    print("\nYour Trading HMM:")
    trading_hmm.display_parameters()

test_exercise_3()

## Summary

### Key Takeaways

1. **HMM Definition**: Five-tuple (S, O, A, B, π) completely specifies the model
2. **Hidden vs Observable**: States are latent; we only see emissions
3. **Three Problems**: Evaluation, Decoding, Learning - each requires specialized algorithms
4. **Probability Constraints**: All parameters must satisfy normalization
5. **Financial Applications**: Natural framework for regime-switching models

### Notation Reference

- **K**: Number of hidden states
- **M**: Number of observation symbols
- **T**: Length of observation sequence
- **q_t**: State at time t
- **o_t**: Observation at time t
- **A_{ij}**: Transition probability from state i to j
- **B_{ij}**: Emission probability of observation j from state i
- **π_i**: Initial probability of state i

### Next Steps

- **Module 1.2**: Simulating HMM sequences
- **Module 2**: The three fundamental algorithms (Forward-Backward, Viterbi, Baum-Welch)
- **Module 3**: Gaussian HMMs for continuous observations
- **Module 4**: Financial applications

---

**Further Reading**:
- Rabiner (1989): "A Tutorial on Hidden Markov Models" - Classic introduction
- Zucchini & MacDonald (2009): "Hidden Markov Models for Time Series"
- Hamilton (1989): "A New Approach to Economic Analysis of Nonstationary Time Series"